# Bengaluru Traffic Digital Twin — Training Notebook
Recreates the original *Topology-Sensitive Traffic Digital Twins for Cross-City Generalization*
pipeline for Bengaluru. Steps:
1. Build zone road graphs (real-junction-anchored)
2. Generate/load zone traffic signals
3. Extract topology features + distance matrix
4. Train BaselineGCN, AdaptiveGNN, TopologyAwareAGNN per training zone
5. Save trained-metric tables for evaluation

**Data note:** OpenStreetMap Overpass API and the reference Kaggle Bengaluru traffic dataset
were not reachable from the build sandbox; road graphs and signals are structured
reconstructions calibrated to each zone's real, documented character. See project README.

In [1]:
import sys, json
sys.path.insert(0, "/home/claude/bengaluru-digital-twin")
import pandas as pd, numpy as np
from data.build_bengaluru_graph import build_all
from data.build_traffic_signals import build_signals
from topology.extract_topology import load_zone_graphs, build_feature_table, topology_distance_matrix
from evaluation.run_experiment import train_model_family, TRAIN_ZONES, TEST_ZONES

## 1. Build road graphs

In [2]:
graphs_nx = build_all()
for zone, G in graphs_nx.items():
    print(f"{zone:16s} {G.number_of_nodes():3d} nodes  {G.number_of_edges():3d} edges")

Koramangala       26 nodes   56 edges
SilkBoard         22 nodes   49 edges
Indiranagar       24 nodes   53 edges
MGRoad_CBD        28 nodes   62 edges
Whitefield        20 nodes   38 edges
ElectronicCity    18 nodes   33 edges
Hebbal            20 nodes   40 edges
Yelahanka         16 nodes   29 edges
Jayanagar         22 nodes   46 edges
Malleshwaram      19 nodes   39 edges


## 2. Generate traffic signals (15-min resolution, 21 days)

In [3]:
signals = build_signals(
    "/home/claude/bengaluru-digital-twin/data/raw/bengaluru_zone_graphs.json",
    "/home/claude/bengaluru-digital-twin/data/raw/bengaluru_traffic_signals.json",
)
print("zones with signals:", list(signals.keys()))

zones with signals: ['Koramangala', 'SilkBoard', 'Indiranagar', 'MGRoad_CBD', 'Whitefield', 'ElectronicCity', 'Hebbal', 'Yelahanka', 'Jayanagar', 'Malleshwaram']


## 3. Topology features + distance matrix

In [4]:
graphs = load_zone_graphs("/home/claude/bengaluru-digital-twin/data/raw/bengaluru_zone_graphs.json")
feat_df = build_feature_table(graphs)
dist_df = topology_distance_matrix(feat_df)
feat_df.round(3)

,N,E,avg_degree,density,clustering_coeff,avg_shortest_path,diameter,betweenness_mean,closeness_mean,spectral_gap
zone,,,,,,,,,,
Koramangala,26,58,4.462,0.178,0.225,2.612,6,0.067,0.392,0.116
SilkBoard,22,48,4.364,0.208,0.274,2.320,5,0.066,0.437,0.172
Indiranagar,24,51,4.250,0.185,0.160,2.406,5,0.064,0.423,0.199
MGRoad_CBD,28,63,4.500,0.167,0.268,2.646,6,0.063,0.387,0.140
Whitefield,20,38,3.800,0.200,0.232,2.589,6,0.088,0.393,0.140
ElectronicCity,18,33,3.667,0.216,0.206,2.549,6,0.097,0.402,0.126
Hebbal,20,40,4.000,0.211,0.135,2.300,4,0.072,0.442,0.228
Yelahanka,16,29,3.625,0.242,0.144,2.217,4,0.087,0.460,0.241
Jayanagar,22,46,4.182,0.199,0.202,2.407,5,0.070,0.426,0.181


In [5]:
dist_df.round(2)

,Koramangala,SilkBoard,Indiranagar,MGRoad_CBD,Whitefield,ElectronicCity,Hebbal,Yelahanka,Jayanagar,Malleshwaram
Koramangala,0.00,4.02,3.57,1.51,4.07,5.38,6.12,8.34,3.57,5.80
SilkBoard,4.02,0.00,3.04,4.50,4.30,5.12,3.97,5.70,1.96,2.79
Indiranagar,3.57,3.04,0.00,4.26,4.44,5.31,2.98,5.69,1.58,3.47
MGRoad_CBD,1.51,4.50,4.26,0.00,5.00,6.46,6.95,9.24,4.40,6.57
Whitefield,4.07,4.30,4.44,5.00,0.00,1.59,5.14,6.02,3.39,4.31
ElectronicCity,5.38,5.12,5.31,6.46,1.59,0.00,5.27,5.43,4.18,4.43
Hebbal,6.12,3.97,2.98,6.95,5.14,5.27,0.00,2.98,2.74,2.30
Yelahanka,8.34,5.70,5.69,9.24,6.02,5.43,2.98,0.00,4.93,3.09
Jayanagar,3.57,1.96,1.58,4.40,3.39,4.18,2.74,4.93,0.00,2.40
Malleshwaram,5.80,2.79,3.47,6.57,4.31,4.43,2.30,3.09,2.40,0.00


## 4. Train BaselineGCN / AdaptiveGNN / TopologyAwareAGNN
Trained per training zone (Koramangala, Silk Board, MG Road/CBD, Indiranagar,
Jayanagar, Malleshwaram, Hebbal). Held-out zones (Whitefield, Electronic City,
Yelahanka) are never seen during training — used later for zero-shot evaluation.

In [6]:
with open("/home/claude/bengaluru-digital-twin/data/raw/bengaluru_traffic_signals.json") as f:
    sig_data = json.load(f)
signals = sig_data["signals"]

trained, seen_metrics = train_model_family(graphs, signals, lookback=8, epochs=40)
seen_metrics.groupby("model")[["MAE","RMSE","MAPE"]].mean().round(3)

/home/claude/bengaluru-digital-twin/models/gnn_models.py:32: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  d_inv_sqrt = np.power(deg, -0.5, where=deg > 0)


,MAE,RMSE,MAPE
model,,,
AdaptiveGNN,101.210,126.914,22.094
BaselineGCN,98.075,123.380,21.874
TopologyAwareAGNN,99.385,124.792,21.650


In [7]:
seen_metrics.to_csv("/home/claude/bengaluru-digital-twin/evaluation/seen_zone_metrics.csv", index=False)
print("Saved seen-zone metrics.")

Saved seen-zone metrics.
